# Gradio Day!

Today we will build User Interfaces using the outrageously simple Gradio framework.

Prepare for joy!

Please note: your Gradio screens may appear in 'dark mode' or 'light mode' depending on your computer settings.

In [98]:
# imports

import os
import requests
from bs4 import BeautifulSoup
from typing import List
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai
import anthropic
import json

In [2]:
import gradio as gr # oh yeah!

In [3]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyDJ


In [4]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

claude = anthropic.Anthropic()

google.generativeai.configure()

In [15]:
# A generic system message - no more snarky adversarial AIs!

system_message = "You are a helpful assistant"

In [6]:
# Let's wrap a call to GPT-4o-mini in a simple function

def message_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    completion = openai.chat.completions.create(
        model='gpt-4o-mini',
        messages=messages,
    )
    return completion.choices[0].message.content

In [7]:
# This can reveal the "training cut off", or the most recent date in the training data

message_gpt("What is today's date?")

"Today's date is October 10, 2023."

## User Interface time!

In [8]:
# here's a simple function

def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [9]:
shout("hello")

Shout has been called with input hello


'HELLO'

In [10]:
그라디오 실행시 백그라운드에서 실행되는 작은 웹서버 실행
port 에서 로컬에서 실행된다

SyntaxError: invalid syntax (1470800432.py, line 1)

In [ ]:
# The simplicty of gradio. This might appear in "light mode" - I'll show you how to make this in dark mode later.

gr.Interface(fn=shout, inputs="textbox", outputs="textbox").launch()

In [ ]:
# Adding share=True means that it can be accessed publically
# A more permanent hosting is available using a platform called Spaces from HuggingFace, which we will touch on next week
# NOTE: Some Anti-virus software and Corporate Firewalls might not like you using share=True. If you're at work on on a work network, I suggest skip this test.

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

In [ ]:
유저 인터페이스는 public 그래디오 웹사이트를 
통해 제공되지만 
코드는 local box에서 실행

In [ ]:
# Adding inbrowser=True opens up a new browser window automatically

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

In [ ]:
퍼블릭 브라우저가 자동으로 열리게 하기

In [ ]:
# Adding inbrowser=True opens up a new browser window automatically

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, share=True)

## Forcing dark mode

Gradio appears in light mode or dark mode depending on the settings of the browser and computer. There is a way to force gradio to appear in dark mode, but Gradio recommends against this as it should be a user preference (particularly for accessibility reasons). But if you wish to force dark mode for your screens, below is how to do it.

In [ ]:
# Define this variable and then pass js=force_dark_mode when creating the Interface

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

In [ ]:
# Inputs and Outputs

view = gr.Interface(
    fn=shout,
    inputs=[gr.Textbox(label="Your message:", lines=6)],
    outputs=[gr.Textbox(label="Response:", lines=8)],
    flagging_mode="never"
)
view.launch()

In [ ]:
# And now - changing the function from "shout" to "message_gpt"

view = gr.Interface(
    fn=message_gpt,
    inputs=[gr.Textbox(label="Your message:", lines=6)],
    outputs=[gr.Textbox(label="Response:", lines=8)],
    flagging_mode="never"
)
view.launch()

In [ ]:
# Let's use Markdown
# Are you wondering why it makes any difference to set system_message when it's not referred to in the code below it?
# I'm taking advantage of system_message being a global variable, used back in the message_gpt function (go take a look)
# Not a great software engineering practice, but quite common during Jupyter Lab R&D!

system_message = "You are a helpful assistant that responds in markdown"

view = gr.Interface(
    fn=message_gpt,
    inputs=[gr.Textbox(label="Your message:")],
    outputs=[gr.Markdown(label="Response:")],  # 응답 마크다운으로
    flagging_mode="never"
)
view.launch()

In [ ]:
그라디오에서 마크다운으로 결과 보여주기 


In [26]:
# Let's create a call that streams back results
# If you'd like a refresher on Generators (the "yield" keyword),
# Please take a look at the Intermediate Python notebook in week1 folder.

def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-4o-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result  

In [ ]:
기존 거 사라지지 않게 하기 위해
result 출력시  누적된거 출력되게 함

In [ ]:
view = gr.Interface(
    fn=stream_gpt,
    inputs=[gr.Textbox(label="Your message:")],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never"
)
view.launch()

In [ ]:
1️⃣ 왜 Gradio UI에서 스트리밍 출력이 되는가?

핵심은 yield 때문이에요.

for chunk in stream:
    result += chunk.choices[0].delta.content or ""
    yield result


openai.chat.completions.create(..., stream=True) → OpenAI API가 답변을 한 덩어리(chunk)씩 계속 보내줍니다.

for chunk in stream: → 이 덩어리들을 순차적으로 받아옵니다.

yield result → 지금까지 쌓인 답변을 즉시 Gradio에 전달합니다.

Gradio는 함수에서 yield가 들어오면 "아, 아직 최종 결과가 아니고 진행 중이구나"라고 이해하고, UI에 중간중간 업데이트를 해줍니다.
즉, return은 딱 한 번만 값을 돌려주지만, yield는 여러 번 내보낼 수 있어서 스트리밍처럼 보이는 효과가 생기는 거예요.

2️⃣ "이게 함수가 아니라 Generator"라는 말의 의미

일반 함수와 제너레이터(generator) 함수의 차이점은 yield에 있습니다.

일반 함수 (return)

호출하면 함수가 끝날 때까지 실행 → 결과를 한 번만 반환

예: def f(): return 10 → f() 실행하면 무조건 10만 돌려줌

제너레이터 함수 (yield)

호출하면 즉시 실행하지 않고, 특별한 **이터레이터 객체(generator)**를 돌려줌

실행할 때마다 yield 위치에서 멈추고, 그 값을 바깥으로 내보냄

다음에 다시 실행하면 멈춘 지점부터 이어서 진행

즉, stream_gpt는 겉보기엔 함수지만, yield를 쓰고 있기 때문에 Generator 함수가 되고,
실제로 stream_gpt("Hello")를 호출하면 → 리스트나 문자열이 아니라 generator 객체가 나옵니다.
Gradio는 이 generator를 소비하면서 값이 나올 때마다 UI에 바로바로 표시하는 거예요.

🔎 비유로 이해하기

return 함수 → "모든 문장을 다 쓴 다음 한꺼번에 보여주는 사람"

yield 함수 → "한 줄 쓰고 보여주고, 또 한 줄 쓰고 보여주는 사람"

그래서 Gradio UI에 글자가 하나씩 스트리밍 되듯이 나타나는 겁니다 ✨

In [ ]:
1️⃣ 제너레이터란?

제너레이터(generator)는 이터레이터(iterator)를 만들어주는 특별한 함수예요.

def로 정의하지만, 안에 yield 키워드를 쓰면 → 그 함수는 제너레이터 함수가 됩니다.

호출하면 즉시 실행되지 않고, 실행 상태를 기억하는 "제너레이터 객체"를 돌려줍니다.

이 객체는 for 루프에서 돌리거나 next()로 하나씩 꺼낼 수 있습니다.

2️⃣ return vs yield 비교
구분	return	yield
반환	딱 한 번 값 반환 후 함수 종료	여러 번 값 반환 가능
실행	함수가 끝날 때까지 실행	yield에서 멈췄다가, 다시 이어서 실행
결과	그냥 값	제너레이터 객체 (이터레이터)
메모리	모든 값을 미리 메모리에 저장	필요할 때마다 하나씩 생성 → 메모리 절약
3️⃣ 간단한 예시
일반 함수
def numbers():
    return [1, 2, 3]

print(numbers())  
# [1, 2, 3]

제너레이터 함수
def numbers():
    yield 1
    yield 2
    yield 3

gen = numbers()
print(gen)  
# <generator object numbers at 0x...>

for n in gen:
    print(n)

# 출력
# 1
# 2
# 3


→ yield를 쓸 때마다 하나씩 값을 돌려주고, 함수는 멈췄다가(for 루프나 next로) 다시 이어서 실행됩니다.

4️⃣ 실무에서 쓰이는 이유

메모리 절약
큰 데이터를 한꺼번에 만들지 않고, 필요할 때마다 조금씩 생성 가능.
예: 10억 개 숫자를 한꺼번에 리스트로 만들면 메모리 터짐 → 제너레이터로는 문제 없음.

def big_numbers():
    for i in range(10**9):
        yield i


스트리밍 처리
네트워크 응답이나 파일 읽기처럼 데이터가 조금씩 들어오는 상황에서 유용.
(→ 지금 쓰신 Gradio 스트리밍 UI가 이 원리!)

파이프라인 처리
여러 단계로 데이터를 가공할 때, 중간 결과를 바로바로 넘겨줄 수 있음.
→ 데이터 흐름을 자연스럽게 연결할 수 있음.

5️⃣ 비유로 이해하기

리스트(return): "도서관에서 책 전체를 한꺼번에 빌려오기"

제너레이터(yield): "책을 한 장씩 복사해서 필요할 때마다 받아오기"

👉 정리하면, **제너레이터는 "필요할 때마다 값 하나씩 뽑아내는 함수"**예요.
그래서 Gradio + OpenAI처럼 "답변이 조금씩 생기는 상황"에 특히 잘 맞습니다.

In [16]:
def stream_claude(prompt):
    result = claude.messages.stream(
        model="claude-3-haiku-20240307",
        max_tokens=1000,
        temperature=0.7,
        system=system_message,
        messages=[
            {"role": "user", "content": prompt},
        ],
    )
    response = ""
    with result as stream:
        for text in stream.text_stream:
            response += text or ""
            yield response

In [17]:
view = gr.Interface(
    fn=stream_claude,
    inputs=[gr.Textbox(label="Your message:")],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [ ]:
제미나이써보기

In [18]:
import google.generativeai as genai

# 환경 변수 또는 직접 키를 세팅
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))



def stream_gemini(prompt):
    model = genai.GenerativeModel("gemini-1.5-flash")  # 또는 "gemini-1.5-pro"

    # system_message는 tools나 safety 설정으로 넣을 수도 있지만,
    # 간단히 user prompt 앞에 붙이는 식으로 처리
    full_prompt = f"{system_message}\n\nUser: {prompt}"

    # streaming 호출
    response = model.generate_content(
        full_prompt,
        stream=True,
        generation_config=genai.types.GenerationConfig(
            max_output_tokens=1000,
            temperature=0.7,
        )
    )

    result = ""
    for chunk in response:
        if chunk.text:   # 토큰이 들어오는 부분
            result += chunk.text
            yield result


In [19]:
view = gr.Interface(
    fn=stream_gemini,
    inputs=[gr.Textbox(label="Your message:")],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## Minor improvement

I've made a small improvement to this code.

Previously, it had these lines:

```
for chunk in result:
  yield chunk
```

There's actually a more elegant way to achieve this (which Python people might call more 'Pythonic'):

`yield from result`

I cover this in more detail in the Intermediate Python notebook in the week1 folder - take a look if you'd like more.

In [20]:
# 톤 프리셋
TONES = {
    "Neutral":        "Answer neutrally, in a clear and concise way.",
    "Friendly":       "Answer in a friendly and encouraging tone.",
    "Formal":         "Answer in a professional and formal style.",
    "Teacher (KOR)":  "한국어로 친절한 멘토처럼 핵심→설명→예시 순서로 설명해줘.",
    "Concise Expert": "Be direct and concise. Use bullet points where possible.",
    "Step-by-step":   "Explain step-by-step with numbered lists."
}

def apply_tone_to_prompt(user_prompt: str, tone_name: str) -> str:
    tone_instruction = TONES.get(tone_name, TONES["Neutral"])
    return f"{tone_instruction}\n\nUser: {user_prompt}"

In [21]:
def stream_model(prompt, model, tone):
     # 1) tone 문구 가져오기
    tone_text = TONES.get(tone, TONES["Neutral"])

     # 2) user 입력 앞에 tone 문구 프리픽스 추가
    prefixed_prompt = f"{tone_text}\n\n{prompt}"
    if model=="GPT":
        result = stream_gpt(prefixed_prompt)
    elif model=="Claude":
        result = stream_claude(prefixed_prompt)
    elif model=="Gemini":
        result = stream_gemini(prefixed_prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [27]:
view = gr.Interface(
    fn=stream_model,
    inputs=[gr.Textbox(label="Your message:"), 
            gr.Dropdown(["GPT", "Claude","Gemini"], label="Select model", value="GPT"),
           gr.Dropdown(list(TONES.keys()), label="Tone", value="Teacher (KOR)")
           ],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


# Building a company brochure generator

Now you know how - it's simple!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you read the next few cells</h2>
            <span style="color:#900;">
                Try to do this yourself - go back to the company brochure in week1, day5 and add a Gradio UI to the end. Then come and look at the solution.
            </span>
        </td>
    </tr>
</table>

In [115]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [116]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [117]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
} 
"""

In [118]:
MODEL = 'gpt-4o-mini'

In [119]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}    ###RESPONSE 포맷 설정 가능, 프롬프트에도 명시해야 동작
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [120]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()  #해당 URL의 본문
    links = get_links(url)  ## 모델 써서 관련있는 링크들 불러와
    print("Found links:", links) # 그거 프린트
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()  #각 링크에 대한 컨텐츠 리절트에 추가
    return result

In [121]:
# With massive thanks to Bill G. who noticed that a prior version of this had a bug! Now fixed.

system_message = "You are an assistant that analyzes the contents of a company website landing page \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in Korean. Respond in markdown."

In [122]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)  # 메인페이지+ 관련 페이지까지 다 1개로 합친 거 
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [123]:
def stream_brochure(company_name, url, model):
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += get_brochure_user_prompt(company_name, url)
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)
    elif model=="Gemini":
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [124]:
view = gr.Interface(
    fn=stream_brochure,
    inputs=[
        gr.Textbox(label="Company name:"),
        gr.Textbox(label="Landing page URL including http:// or https://"),
        gr.Dropdown(["GPT", "Claude","Gemini"], label="Select model")],
    outputs=[gr.Markdown(label="Brochure:")],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}
